In [41]:
import os
import re
from collections import defaultdict

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)

# ---- Settings ----
USDA_DIR = "../data/raw/FoodData_Central_sr_legacy_food_csv_2018-04"   
INGREDIENTS_CSV = "ingredients_to_map.csv"                  # notebook 2 outcome
OVERRIDES_CSV = "manual_overrides.csv"                      # manula correctionsة
TOP_REVIEW = 100                                            


def read_usda(name, **kw):
    path = os.path.join(USDA_DIR, name)
    assert os.path.exists(path), f"Could not find {path}"
    return pd.read_csv(path, **kw)

In [42]:
food = read_usda("food.csv")
nutrient = read_usda("nutrient.csv")
food_nutrient = read_usda("food_nutrient.csv", usecols=["fdc_id", "nutrient_id", "amount"])

if "data_type" in food.columns:
    print(food["data_type"].value_counts())
    food = food[food["data_type"] == "sr_legacy_food"]

food = food[["fdc_id", "description"]].drop_duplicates("fdc_id").reset_index(drop=True)
nutrient["name"] = nutrient["name"].astype(str).str.strip()
fn = food_nutrient[food_nutrient["fdc_id"].isin(food["fdc_id"])].copy()

print(f"foods: {len(food):,} | nutrients: {len(nutrient):,} | food_nutrient rows: {len(fn):,}")
food.head()

data_type
sr_legacy_food    7793
Name: count, dtype: int64
foods: 7,793 | nutrients: 474 | food_nutrient rows: 644,125


,fdc_id,description
0,167512,"Pillsbury Golden Layer Buttermilk Biscuits, Artificial Flavor, ref..."
1,167513,"Pillsbury, Cinnamon Rolls with Icing, refrigerated dough"
2,167514,"Kraft Foods, Shake N Bake Original Recipe, Coating for Pork, dry"
3,167515,"George Weston Bakeries, Thomas English Muffins"
4,167516,"Waffles, buttermilk, frozen, ready-to-heat"


In [43]:
# المفتاح: (نمط الاسم، الوحدة)
NUTRIENT_SPEC = {
    "kcal":         (r"^energy$", "KCAL"),
    "protein_g":    (r"^protein$", "G"),
    "fat_g":        (r"^total lipid \(fat\)$", "G"),
    "carbs_g":      (r"^carbohydrate, by difference$", "G"),
    "fiber_g":      (r"^fiber, total dietary$", "G"),
    "calcium_mg":   (r"^calcium, ca$", "MG"),
    "iron_mg":      (r"^iron, fe$", "MG"),
    "magnesium_mg": (r"^magnesium, mg$", "MG"),
    "potassium_mg": (r"^potassium, k$", "MG"),
    "sodium_mg":    (r"^sodium, na$", "MG"),
    "zinc_mg":      (r"^zinc, zn$", "MG"),
    "vitc_mg":      (r"^vitamin c, total ascorbic acid$", "MG"),
    "folate_ug":    (r"^folate, total$", "UG"),
    "vitb6_mg":     (r"^vitamin b-6$", "MG"),
    "vitb12_ug":    (r"^vitamin b-12$", "UG"),
}

counts = fn.groupby("nutrient_id").size()
unit_up = nutrient["unit_name"].astype(str).str.upper()

resolved, rows = {}, []
for key, (pat, unit) in NUTRIENT_SPEC.items():
    m = nutrient[nutrient["name"].str.contains(pat, case=False, regex=True) & (unit_up == unit)].copy()
    if m.empty:
    print(f"!! Nutrient {key} not found (pattern: {pat}, unit: {unit}). Search with nutrient[nutrient.name.str.contains('...')]")        continue
    m["n_foods"] = m["id"].map(counts).fillna(0)
    best = m.sort_values("n_foods", ascending=False).iloc[0]
    resolved[key] = int(best["id"])
    rows.append({"key": key, "nutrient_id": int(best["id"]), "name": best["name"],
                 "unit": best["unit_name"], "nutrient_nbr": best.get("nutrient_nbr"),
                 "foods_with_value": int(best["n_foods"])})

pd.DataFrame(rows)

,key,nutrient_id,name,unit,nutrient_nbr,foods_with_value
0,kcal,1008,Energy,KCAL,208.0,7793
1,protein_g,1003,Protein,G,203.0,7793
2,fat_g,1004,Total lipid (fat),G,204.0,7793
3,carbs_g,1005,"Carbohydrate, by difference",G,205.0,7793
4,fiber_g,1079,"Fiber, total dietary",G,291.0,7231
5,calcium_mg,1087,"Calcium, Ca",MG,301.0,7708
6,iron_mg,1089,"Iron, Fe",MG,303.0,7713
7,magnesium_mg,1090,"Magnesium, Mg",MG,304.0,7421
8,potassium_mg,1092,"Potassium, K",MG,306.0,7516
9,sodium_mg,1093,"Sodium, Na",MG,307.0,7709


In [44]:
ids = list(resolved.values())
wide = (fn[fn["nutrient_id"].isin(ids)]
        .pivot_table(index="fdc_id", columns="nutrient_id", values="amount", aggfunc="first")
        .rename(columns={v: k for k, v in resolved.items()}))

# If calories are missing for a food, try Atwater (2047, then 2048)
alt = (fn[fn["nutrient_id"].isin([2047, 2048])]
       .pivot_table(index="fdc_id", columns="nutrient_id", values="amount", aggfunc="first"))
if "kcal" in wide.columns:
    for c in (2047, 2048):
        if c in alt.columns:
            wide["kcal"] = wide["kcal"].fillna(alt[c].reindex(wide.index))

per100 = food.set_index("fdc_id").join(wide)
per100.to_pickle("usda_nutrients_per100g.pkl")

print(f"per100 table: {per100.shape}")
print("\nMissing % per nutrient (NaN = not reported in USDA, not zero):")
print((per100[list(resolved)].isna().mean() * 100).round(1))
per100.head()

per100 table: (7793, 16)

مفقود % لكل مغذي (NaN = ما انذكر بـ USDA، مش صفر):
kcal             0.0
protein_g        0.0
fat_g            0.0
carbs_g          0.0
fiber_g          7.2
calcium_mg       1.1
iron_mg          1.0
magnesium_mg     4.8
potassium_mg     3.6
sodium_mg        1.1
zinc_mg          5.0
vitc_mg          5.9
folate_ug       12.1
vitb6_mg         6.8
vitb12_ug        8.7
dtype: float64


,description,protein_g,fat_g,carbs_g,kcal,fiber_g,calcium_mg,iron_mg,magnesium_mg,potassium_mg,sodium_mg,zinc_mg,vitc_mg,vitb6_mg,folate_ug,vitb12_ug
fdc_id,,,,,,,,,,,,,,,,
167512,"Pillsbury Golden Layer Buttermilk Biscuits, Artificial Flavor, ref...",5.88,13.24,41.18,307.0,1.2,NaN,2.12,NaN,NaN,1059.0,NaN,NaN,NaN,NaN,NaN
167513,"Pillsbury, Cinnamon Rolls with Icing, refrigerated dough",4.34,11.27,53.42,330.0,1.4,28.0,1.93,NaN,NaN,780.0,NaN,0.1,NaN,NaN,NaN
167514,"Kraft Foods, Shake N Bake Original Recipe, Coating for Pork, dry",6.10,3.70,79.80,377.0,NaN,NaN,NaN,NaN,NaN,2182.0,NaN,NaN,NaN,NaN,NaN
167515,"George Weston Bakeries, Thomas English Muffins",8.00,1.80,46.00,232.0,NaN,180.0,1.40,NaN,NaN,345.0,NaN,NaN,NaN,40.0,NaN
167516,"Waffles, buttermilk, frozen, ready-to-heat",6.58,9.22,41.05,273.0,2.2,279.0,6.04,19.0,126.0,621.0,0.48,0.0,0.972,63.0,2.86


In [45]:
STOP = {"and", "with", "or", "the", "in", "of", "for", "from", "by", "a", "an", "to", "as", "not", "no", "made"}
KEEP = {"molasses", "asparagus", "hummus", "couscous", "swiss", "grits", "citrus",
        "lemongrass", "watercress", "cress", "bass", "hops", "less"}
SPECIAL = {"chilies": "chili", "cookies": "cookie", "brownies": "brownie", "leaves": "leaf",
           "loaves": "loaf", "halves": "half", "tomatoes": "tomato", "potatoes": "potato", "peas": "pea"}


def singular(w):
    if w in SPECIAL:
        return SPECIAL[w]
    if w in KEEP or len(w) <= 3:
        return w
    if w.endswith("ies"):
        return w[:-3] + "y"
    if w.endswith(("ches", "shes", "xes")):
        return w[:-2]
    if w.endswith("s") and not w.endswith(("ss", "us", "is")):
        return w[:-1]
    return w


def toks(s):
    s = re.sub(r"[^a-z0-9\s]", " ", str(s).lower())
    return [singular(t) for t in s.split() if t not in STOP]



# USDA sometimes starts the description with the category name ("Spices, garlic powder", "Nuts, walnuts, english"),
# so these words aren't the actual food name. We treat the food name as the part after the category.
CATEGORY = {"spice", "oil", "nut", "seed", "cheese", "cereal", "beverage", "fish", "pork", "beef",
            "chicken", "turkey", "lamb", "veal", "soup", "sauce", "leavening", "agent", "candy",
            "snack", "sugar", "sweetener", "milk", "cream", "egg", "crustacean", "mollusk",
            "game", "meat", "syrup", "dressing", "salad", "fat", "spread"}
# Words that indicate a derived product (oil, flour, dried...): we penalize them if the ingredient doesn't contain the same word
DERIV = {"oil", "juice", "flour", "powder", "extract", "puree", "paste", "candy", "bran", "syrup",
         "spread", "cookie", "cracker", "cake", "bread", "imitation", "substitute", "meatless",
         "tofu", "roll", "analog", "drink", "mix", "flavored", "formula", "snack", "bar", "dessert",
         "pudding", "pie", "sandwich", "biscuit", "dough", "gravy", "roasted", "smoked", "pickled",
         "canned", "dried", "dehydrated", "sweetened", "babyfood", "reduced", "hydrolyzed"}
STRONG = {"meatless", "imitation", "substitute", "analog", "tofu", "candy", "formula", "flavored", "babyfood"}
BAD = ("babyfood", "baby food", "restaurant", "fast foods", "commodity", "school lunch", "infant formula")
BRANDS = ("pillsbury", "kraft", "nestle", "george weston", "mcdonald", "burger king", "kellogg",
          "general mills", "campbell", "pepperidge", "betty crocker", "keebler", "nabisco", "hershey",
          "oscar mayer", "hormel", "jimmy dean", "tyson", "gerber")
# Synonyms (our ingredient name -> the name used in USDA)
SYNONYMS = {"ketchup": "catsup", "zest": "peel", "mint": "spearmint"}


def syn(q):
    q = str(q).lower()
    for a, b in SYNONYMS.items():
        q = re.sub(rf"\b{a}\b", b, q)
    return q


descs = food["description"].tolist()
fdc_ids = food["fdc_id"].tolist()
index_data = []
inv = defaultdict(list)
for i, d in enumerate(descs):
    segs = [set(toks(x)) for x in d.split(",")[:4]]
    S1 = segs[0] if segs else set()
    if S1 and S1 <= CATEGORY and len(segs) > 1:
        name = set().union(*segs[1:3])
    else:
        name = S1
    dt = set(toks(d))
    brand = bool(re.search(r"\b[A-Z]{3,}\b", d)) or any(b in d.lower() for b in BRANDS)
    index_data.append((S1, name, dt, brand))
    for w in dt:
        inv[w].append(i)


def score_candidates(query, top=5):
    qt = toks(syn(query))
    if not qt:
        return []
    qset, head = set(qt), qt[-1]
    want_cooked = "cooked" in qset
    cand = set()
    for w in qset:
        cand.update(inv.get(w, ()))
    out = []
    for i in cand:
        S1, name, dt, brand = index_data[i]
        cov = len(qset & dt) / len(qset)
        head_ok = 1.0 if head in (name | S1) else 0.0
        name_cov = len(qset & (name | S1)) / len(qset)
        extras = min(len(name - qset), 3)
        deriv = min(len((dt & DERIV) - qset), 2)
        strong = 1 if ((dt & STRONG) - qset) else 0
        s = cov + 0.6 * head_ok + 0.3 * name_cov - 0.01 * len(dt) - 0.15 * extras - 0.3 * deriv - 0.8 * strong
        if want_cooked:
            s += 0.15 if "cooked" in dt else -0.1
        else:
            s += 0.15 if "raw" in dt else 0.0
            s -= 0.1 if "cooked" in dt else 0.0
        dl = descs[i].lower()
        if any(b in dl for b in BAD):
            s -= 0.5
        if brand:
            s -= 0.5
        out.append((fdc_ids[i], descs[i], round(s, 3), cov, head_ok, len(dt)))
    out.sort(key=lambda x: (-x[2], x[5]))
    return out[:top]


def search_usda(query, n=10):
    res = score_candidates(query, top=n)
    return pd.DataFrame(res, columns=["fdc_id", "description", "score", "coverage", "head_ok", "n_tokens"])


# test
search_usda("garlic powder", 5)

,fdc_id,description,score,coverage,head_ok,n_tokens
0,171325,"Spices, garlic powder",1.87,1.0,1.0,3
1,167806,Baobab powder,1.08,0.5,1.0,2
2,170461,Tomato powder,1.08,0.5,1.0,2
3,174858,"Beverages, Lemonade, powder",1.07,0.5,1.0,3
4,171319,"Spices, chili powder",1.07,0.5,1.0,3


In [46]:
search_usda("cranberry sauce canned")

,fdc_id,description,score,coverage,head_ok,n_tokens
0,172880,"Sauce, pizza, canned, ready-to-serve",1.267,0.666667,1.0,5
1,173961,"Cranberry sauce, canned, sweetened",0.860,1.000000,0.0,4
2,171110,"Chicken, canned, no broth",0.853,0.333333,1.0,3
3,172882,"Soup, tomato, canned, condensed",0.843,0.333333,1.0,4
4,171153,"Soup, minestrone, canned, condensed",0.843,0.333333,1.0,4
5,171541,"Soup, cheese, canned, condensed",0.843,0.333333,1.0,4
6,171551,"Soup, onion, canned, condensed",0.843,0.333333,1.0,4
7,172194,"Milk, canned, evaporated, with added vitamin A",0.833,0.333333,1.0,5
8,174538,"Soup, vegetable, canned, low sodium, condensed",0.823,0.333333,1.0,6
9,171145,"Soup, chicken, canned, chunky, ready-to-serve",0.823,0.333333,1.0,6


In [47]:
set_override("whole berry cranberry sauce", 173961)

saved: 'whole berry cranberry sauce' -> 173961 (Cranberry sauce, canned, sweetened)


In [48]:
ing = pd.read_csv(INGREDIENTS_CSV)
need = [c for c in ["rank", "ingredient", "n_recipes"] if c not in ing.columns]
assert not need, f"أعمدة ناقصة بـ {INGREDIENTS_CSV}: {need}"
if "raw_variants" not in ing.columns:
    ing["raw_variants"] = ""
ing = ing[["rank", "ingredient", "n_recipes", "raw_variants"]].drop_duplicates("ingredient").reset_index(drop=True)

auto_rows = []
for r in ing.itertuples():
    c = score_candidates(r.ingredient, top=3)
    row = {"ingredient": r.ingredient}
    for k in range(3):
        if k < len(c):
            row[f"cand{k+1}_id"], row[f"cand{k+1}_desc"], row[f"cand{k+1}_score"] = c[k][0], c[k][1], c[k][2]
        else:
            row[f"cand{k+1}_id"], row[f"cand{k+1}_desc"], row[f"cand{k+1}_score"] = None, "", np.nan
    if c:
        cov1, head1 = c[0][3], c[0][4]
        margin = c[0][2] - (c[1][2] if len(c) > 1 else -9)
        if cov1 == 1 and head1 and margin >= 0.1:
            row["confidence"] = "high"
        elif cov1 >= 0.75 and head1:
            row["confidence"] = "medium"
        else:
            row["confidence"] = "low"
    else:
        row["confidence"] = "none"
    auto_rows.append(row)

auto = ing.merge(pd.DataFrame(auto_rows), on="ingredient")
auto["confidence"].value_counts()

confidence
high      195
low       155
medium    132
none       18
Name: count, dtype: int64

In [49]:
HINTS = {
    "salt": "Salt, table",
    "sugar": "Sugars, granulated",
    "brown sugar": "Sugars, brown",
    "powdered sugar": "Sugars, powdered",
    "butter": "Butter, salted",
    "egg": "Egg, whole, raw, fresh",
    "milk": "Milk, whole, 3.25% milkfat, with added vitamin D",
    "water": "Beverages, water, tap, drinking",
    "all purpose flour": "Wheat flour, white, all-purpose, enriched, bleached",
    "olive oil": "Oil, olive, salad or cooking",
    "baking soda": "Leavening agents, baking soda",
    "vanilla extract": "Vanilla extract",
    "black pepper": "Spices, pepper, black",
    "cinnamon": "Spices, cinnamon, ground",
    "garlic": "Garlic, raw",
    "onion": "Onions, raw",
    "lemon juice": "Lemon juice, raw",
    "spinach": "Spinach, raw",
    "carrot": "Carrots, raw",
    "honey": "Honey",
    "heavy cream": "Cream, fluid, heavy whipping",
    "sour cream": "Cream, sour, cultured",
    "cream cheese": "Cheese, cream",
    "broccoli": "Broccoli, raw",
    "kale": "Kale, raw",
    "lentil": "Lentils, raw",
    # Corrections after reviewing the first results
    "baking powder": "Leavening agents, baking powder, double-acting, sodium aluminum sulfate",
    "tomato": "Tomatoes, red, ripe, raw, year round average",
    "garlic powder": "Spices, garlic powder",
    "chili powder": "Spices, chili powder",
    "walnut": "Nuts, walnuts, english",
    "mayonnaise": "Salad dressing, mayonnaise, regular",
    "chicken breast": "Chicken, broilers or fryers, breast, meat only, raw",
    "potato": "Potatoes, flesh and skin, raw",
    "cayenne pepper": "Spices, pepper, red or cayenne",
    "rolled oat": "Cereals, oats, regular and quick, not fortified, dry",
    "buttermilk": "Milk, buttermilk, fluid, cultured, lowfat",
    "soy sauce": "Soy sauce made from soy and wheat (shoyu)",
    "lemon zest": "Lemon peel, raw",
    "green onion": "Onions, spring or scallions (includes tops and bulb), raw",
    "cheddar cheese": "Cheese, cheddar",
    "apple": "Apples, raw, with skin",
    "mushroom": "Mushrooms, white, raw",
    "pumpkin puree": "Pumpkin, canned, without salt",
    "bacon": "Pork, cured, bacon, cooked, broiled, pan-fried or roasted",
    "chicken broth": "Soup, chicken broth, ready-to-serve",
    "mint": "Spearmint, fresh",
    "ketchup": "Catsup",
}

by_desc = food.drop_duplicates("description").set_index("description")["fdc_id"]
hint_ids = {}
missing_hints = []
for ingredient, d in HINTS.items():
    if d in by_desc.index:
        hint_ids[ingredient] = int(by_desc[d])
    else:
        missing_hints.append((ingredient, d))
print(f"Hints applied: {len(hint_ids)} | hints whose USDA name was not found: {len(missing_hints)}")
for ingredient, d in missing_hints:
    print(f"   {ingredient!r}: {d!r}")


def set_override(ingredient, fdc_id, note=""):
    ov = pd.read_csv(OVERRIDES_CSV) if os.path.exists(OVERRIDES_CSV) else pd.DataFrame(columns=["ingredient", "fdc_id", "note"])
    ov = ov[ov["ingredient"] != ingredient]
    ov = pd.concat([ov, pd.DataFrame([{"ingredient": ingredient, "fdc_id": int(fdc_id), "note": note}])], ignore_index=True)
    ov.to_csv(OVERRIDES_CSV, index=False, encoding="utf-8-sig")
    d = food.loc[food["fdc_id"] == int(fdc_id), "description"]
    print(f"saved: {ingredient!r} -> {int(fdc_id)} ({d.iloc[0] if len(d) else 'fdc_id doesnt exist!'})")

Hints applied: 44 | hints whose USDA name was not found: 4
   'chicken breast': 'Chicken, broilers or fryers, breast, meat only, raw'
   'cheddar cheese': 'Cheese, cheddar'
   'apple': 'Apples, raw, with skin'
   'bacon': 'Pork, cured, bacon, cooked, broiled, pan-fried or roasted'


In [50]:
# apply: manual > hint > auto
if os.path.exists(OVERRIDES_CSV):
    overrides = pd.read_csv(OVERRIDES_CSV)
else:
    overrides = pd.DataFrame(columns=["ingredient", "fdc_id", "note"])
    overrides.to_csv(OVERRIDES_CSV, index=False, encoding="utf-8-sig")
manual_ids = dict(zip(overrides["ingredient"], overrides["fdc_id"].astype(int)))

final_map = auto.copy()
final_map["fdc_id"] = final_map["cand1_id"]
final_map["source"] = "auto"

hint_mask = final_map["ingredient"].isin(hint_ids)
final_map.loc[hint_mask, "fdc_id"] = final_map.loc[hint_mask, "ingredient"].map(hint_ids)
final_map.loc[hint_mask, "source"] = "hint"
final_map.loc[hint_mask, "confidence"] = "high"

manual_mask = final_map["ingredient"].isin(manual_ids)
final_map.loc[manual_mask, "fdc_id"] = final_map.loc[manual_mask, "ingredient"].map(manual_ids)
final_map.loc[manual_mask, "source"] = "manual"
final_map.loc[manual_mask, "confidence"] = "manual"

desc_by_id = food.set_index("fdc_id")["description"]
final_map["fdc_id"] = pd.to_numeric(final_map["fdc_id"], errors="coerce").astype("Int64")
final_map["usda_description"] = final_map["fdc_id"].map(desc_by_id)

cols = ["rank", "ingredient", "n_recipes", "fdc_id", "usda_description", "source", "confidence",
        "cand1_id", "cand1_desc", "cand1_score", "cand2_id", "cand2_desc", "cand2_score",
        "cand3_id", "cand3_desc", "cand3_score", "raw_variants"]
final_map = final_map[cols].sort_values("rank").reset_index(drop=True)
final_map.to_csv("ingredient_usda_map.csv", index=False, encoding="utf-8-sig")

review = final_map[(final_map["confidence"].isin(["low", "medium", "none"])) |
                   ((final_map["rank"] <= TOP_REVIEW) & (final_map["source"] != "manual"))]
review = review.sort_values("n_recipes", ascending=False)
review.to_csv("review_queue.csv", index=False, encoding="utf-8-sig")

print(final_map["source"].value_counts().to_dict())
print(final_map["confidence"].value_counts().to_dict())
print(f"\nreview_queue.csv: {len(review)} ingredients")

{'auto': 445, 'hint': 44, 'manual': 11}
{'high': 215, 'low': 151, 'medium': 112, 'manual': 11, 'none': 11}

review_queue.csv: 347 ingredients


In [51]:
view = final_map[["rank", "ingredient", "n_recipes", "usda_description", "source", "confidence"]]
view.head(60)

,rank,ingredient,n_recipes,usda_description,source,confidence
0,1,salt,24070,"Salt, table",hint,high
1,2,butter,19213,"Butter, salted",hint,high
2,3,sugar,17759,"Sugars, granulated",hint,high
3,4,all purpose flour,16346,"Wheat flour, white, all-purpose, enriched, bleached",hint,high
4,5,egg,15584,"Egg, whole, raw, fresh",hint,high
5,6,black pepper,8047,"Spices, pepper, black",hint,high
6,7,water,8156,"Beverages, water, tap, drinking",hint,high
7,8,onion,7223,"Onions, raw",hint,high
8,9,garlic,6802,"Garlic, raw",hint,high
9,10,vanilla extract,9588,Vanilla extract,hint,high


In [52]:
# Review by confidence level: lowest confidence among the most frequently used ingredients
final_map[final_map["confidence"].isin(["low", "none", "medium"])] \
    .sort_values("n_recipes", ascending=False) \
    [["rank", "ingredient", "n_recipes", "usda_description", "cand2_desc", "confidence"]].head(40)

,rank,ingredient,n_recipes,usda_description,cand2_desc,confidence
18,19,margarine,3905,"Margarine, regular, hard, soybean (hydrogenated)","Margarine, regular, 80% fat, composite, stick, with salt",medium
21,22,parmesan cheese,2322,"Cheese, parmesan, grated","Cheese, parmesan, hard",medium
42,43,shortening,2092,"Shortening, vegetable, household, composite","Shortening, confectionery, fractionated palm",medium
38,39,apple,1743,"Apples, raw, without skin","Apples, raw, golden delicious, with skin",medium
47,48,white vinegar,1531,"Vinegar, distilled","Vinegar, balsamic",low
52,53,raisin,1518,"Raisins, seeded","Raisins, golden, seedless",medium
46,47,bacon,1462,"Bacon, turkey, microwaved","Bacon, turkey, unprepared",medium
34,35,green bell pepper,1440,"Peppers, sweet, green, raw","Peppers, hot chili, green, raw",low
70,71,peanut butter,1167,"Peanut butter, chunk style, with salt","Sauce, peanut, made from peanut butter, water, soy sauce",medium
75,76,chocolate chip,1157,"Tortilla chips, yellow, plain, salted","Tortilla chips, low fat, baked without fat",low


In [53]:
chk = final_map.merge(per100.reset_index(), left_on="fdc_id", right_on="fdc_id", how="left")
key_cols = [c for c in ["kcal", "iron_mg", "vitc_mg", "magnesium_mg", "protein_g"] if c in chk.columns]

print("Number of ingredients with missing nutrient values (NaN) in the selected food:")
print(chk[key_cols].isna().sum())

top_missing = chk[chk[key_cols].isna().any(axis=1) & (chk["rank"] <= 150)]
top_missing[["rank", "ingredient", "usda_description"] + key_cols]

عدد المكونات اللي غذائياً ناقصة (NaN) بالأكلة المختارة:
kcal            11
iron_mg         11
vitc_mg         25
magnesium_mg    21
protein_g       11
dtype: int64


,rank,ingredient,usda_description,kcal,iron_mg,vitc_mg,magnesium_mg,protein_g
110,111,shrimp,"Crustaceans, shrimp, raw",85.0,0.52,NaN,35.0,20.1


In [54]:
fp = read_usda("food_portion.csv")
mu = read_usda("measure_unit.csv")
print(fp.head(8))
print(mu.head(10))

      id  fdc_id  seq_num  amount  measure_unit_id  portion_description                   modifier  gram_weight  data_points  footnote  min_year_acquired
0  81549  167512        1     1.0             9999                  NaN                    serving         34.0          NaN       NaN                NaN
1  81550  167513        1     1.0             9999                  NaN  serving 1 roll with icing         44.0          1.0       NaN                NaN
2  81551  167514        1     1.0             9999                  NaN                    serving         28.0          NaN       NaN                NaN
3  81552  167515        1     1.0             9999                  NaN                    serving         57.0          NaN       NaN                NaN
4  81553  167516        1     1.0             9999                  NaN             waffle, square         39.0         10.0       NaN                NaN
5  81554  167516        2     1.0             9999                  NaN     

In [55]:
need_cols = {"fdc_id", "gram_weight"}
if need_cols <= set(fp.columns):
    mu_name = fp["measure_unit_id"].map(mu.set_index("id")["name"]) if "measure_unit_id" in fp.columns else ""
    pieces = []
    for series in (mu_name,
                   fp["portion_description"] if "portion_description" in fp.columns else "",
                   fp["modifier"] if "modifier" in fp.columns else ""):
        s = pd.Series(series, index=fp.index).fillna("").astype(str).str.strip()
        s = s.where(s.ne("undetermined"), "")
        pieces.append(s)
    unit_s, desc_s, mod_s = pieces
    dup = pd.Series([bool(u) and m.startswith(u) for u, m in zip(unit_s, mod_s)], index=unit_s.index)
    unit_s = unit_s.where(~dup, "")
    fp["portion_label"] = (unit_s + " " + desc_s + " " + mod_s).str.replace(r"\s+", " ", regex=True).str.strip()

    chosen = final_map[["ingredient", "fdc_id"]].dropna()
    portions = chosen.merge(fp, on="fdc_id", how="left")
    keep = [c for c in ["ingredient", "fdc_id", "seq_num", "amount", "portion_label", "gram_weight"] if c in portions.columns]
    portions = portions[keep]
    portions.to_csv("ingredient_portions.csv", index=False, encoding="utf-8-sig")

    n_with = portions.dropna(subset=["gram_weight"])["ingredient"].nunique()
    print(f"Ingredients with at least one household measure weight: {n_with} of {chosen['ingredient'].nunique()}")
    print("\nMost common portion labels:")
    print(portions["portion_label"].value_counts().head(15))
else:
    print("Unexpected food_portion columns:", fp.columns.tolist())

مكونات إلها وزن منزلي واحد على الأقل: 479 من 489

أكتر تسميات الأوزان شيوعاً:
portion_label
cup              167
tbsp             131
tsp               87
oz                81
fl oz             33
cup, chopped      23
NLEA serving      18
cubic inch        18
small             12
cup, diced        12
cup, shredded     12
large             11
tsp, ground       11
medium            10
cup, sliced       10
Name: count, dtype: int64
